# Merge Oversegmentation GNN — train / CV

Relink micro-SAM AIS **cell-fragment masks** that belong to the same biological cell
(the AIS model oversegments long hyphae). Nodes = AIS fragments; candidate edges = kNN
by boundary distance; positive edges = adjacent fragments of the same GT cell (per-cell
MST). Whole cells are recovered at inference by connected components over predicted
positives.

Pipeline docs: `scene_graph_network/gnn_documentation/Cell Mask Graph Data Flow.md`.
Modeled on `4_Visual GNN Test.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Load images, AIS predictions, and GT masks

Fill in the parallel path lists below (one entry per image). The DIC channel of the raw
image drives the edge intensity features; the GT whole-cell masks drive the merge labels.

In [ ]:
import json
from pathlib import Path

# === USER CONFIG ===
# Data lives in a standalone JSON so it can be edited without touching the
# notebook, and shared with the micro-SAM finetune config's layout. Per sample:
#   image  - one file per channel; ALL listed channels are summed into the
#            intensity image driving the node/edge features, so list only
#            channels that carry signal
#   ais    - micro-SAM AIS instance label map (the graph nodes). Required.
#   label  - ground-truth whole-cell instance map. Empty -> no training labels.
#   embedding - saved micro-sam tiled embedding zarr. Empty -> no visual branch.
DATA_CONFIG = "merge_oversegmentation_data.json"

with open(DATA_CONFIG) as f:
    SAMPLES = json.load(f)["samples"]

MICROSAM_ENV = "microsam"   # conda env that can read the v3 embedding zarr and stitch it

K = 10  # kNN candidate edges per fragment, by boundary-to-boundary distance.
        # Raised from 7: k=7 left 22 of 483 true merges (4.6%) outside the candidate
        # set, where no model can ever predict them; k=10 leaves 6 (1.2%). Matches
        # 12_Node Type GNN.ipynb so the two are a baseline/treatment pair.

# Where to persist the built PyG dataset (and any stitched embeddings).
DATASET_ROOT = "/home/wangchuyao/Projects/C_Albicans Thesis Project/7. Data/graph_datasets/merge_oversegmentation"

missing = [i for i, s in enumerate(SAMPLES) if not s.get("ais")]
if missing:
    raise ValueError(
        f"Samples {missing} have no `ais` path. The AIS label map is the node set "
        f"and is required -- fill it in {DATA_CONFIG}."
    )
print(f"{len(SAMPLES)} samples from {DATA_CONFIG}")
for i, s in enumerate(SAMPLES):
    print(f"  {i}: channels={list(s['image'])} "
          f"label={'yes' if s.get('label') else 'NO'} "
          f"embedding={'yes' if s.get('embedding') else 'NO'} "
          f"shift={s.get('correct_DIC_shift', [0, 0])}")


In [ ]:
import numpy as np
import tifffile

from image_processing_tools.image_class.image_container import ImageContainer


def _config(correct_dic_shift):
    """Per-sample preprocessing config.

    `resize_image: False` is load-bearing twice over: it keeps the intensity
    image pixel-aligned with `ais_labels`, and it routes
    `get_image_for_processing()` to `_get_high_contrast_16bit()` so every channel
    is percentile-clipped (0.35 / 99.65) and stretched before it is summed.
    """
    return {
        "preprocessing": {
            "resize_image": False,
            "max_dim": 1080,
            "outlier_percentile": 0.35,
            "quantization": "16bit",
            "correct_DIC_shift": correct_dic_shift,
        }
    }


def load_label_map(path):
    """Read an instance label map as int32 (H, W)."""
    lab = np.squeeze(tifffile.imread(str(Path(path).expanduser())))
    assert lab.ndim == 2, f"Expected a 2D label map, got shape {lab.shape} for {path}"
    return lab.astype(np.int32)


ais_labels, gt_labels, intensity_images, display_images = [], [], [], []
for s in SAMPLES:
    channels = [Path(p).expanduser() for p in s["image"].values()]
    config = _config(s.get("correct_DIC_shift", [0, 0]))

    # Grouping the channels in the structure makes the constructor reduce them
    # via _sum_channels -> one 2D channel, summed and stretched to full range.
    # A single channel passes through unchanged, so mono and multi-channel
    # samples land on the same 0..65535 scale.
    intensity_images.append(ImageContainer([channels], config).merge())

    # Ungrouped, the channels stay separate: merge() combines them into (H, W, C)
    # for the prediction overlays. plot_edge_predictions dispatches on the
    # channel count.
    display_images.append(ImageContainer(channels, config).merge())

    ais_labels.append(load_label_map(s["ais"]))
    gt_labels.append(load_label_map(s["label"]) if s.get("label") else None)

print(f"Loaded {len(ais_labels)} images.")
for i, (a, g, inten, disp) in enumerate(zip(ais_labels, gt_labels, intensity_images, display_images)):
    assert inten.ndim == 2, f"img {i}: intensity image must be 2D, got {inten.shape}"
    assert inten.shape == a.shape, (
        f"img {i}: intensity {inten.shape} is not aligned with AIS labels {a.shape}"
    )
    n_gt = "no GT" if g is None else f"GT cells={int(g.max())}"
    print(f"  img {i}: AIS fragments={int(a.max())}, {n_gt}, "
          f"shape={a.shape}, intensity={inten.min()}..{inten.max()}, display={disp.shape}")


## 2. Make the graph

`build_cell_graph_data` runs `extract_cell_graph` (fragment nodes, kNN edges, 8 node +
10 edge features), derives per-cell MST merge labels from the GT masks, and assembles one
PyG `Data`. If a sample's `embedding` is set, the saved micro-sam tiled store is stitched and
attached for the visual branch.

Micro-sam writes the embedding store in the env that ran AIS; if that is a zarr **v3** store and this kernel's zarr is too old to read it, `load_or_stitch_embeddings` automatically stitches it in the `MICROSAM_ENV` conda env (via `conda run`) and caches the result as an `.npz`.


In [ ]:
from image_processing_tools.scene_graph_network.build_cell_dataset import build_cell_graph_data
from image_processing_tools.scene_graph_network.precompute_microsam_feats import load_or_stitch_embeddings

emb_dir = Path(DATASET_ROOT) / "embeddings"


def _embedding_path(sample):
    """Absolute path to a sample's tiled .zarr store, or None.

    Paths in the JSON are `~`-relative; zarr does no tilde expansion of its own
    and would look for a directory literally named "~".
    """
    emb = sample.get("embedding")
    return Path(emb).expanduser() if emb else None


def _npz_name(embedding_path):
    """Cache name for a stitched embedding, derived from the store it came from.

    Keyed on the .zarr's own name, never on the sample's position in SAMPLES:
    the JSON can be reordered or extended, and an index-keyed cache would then
    hand a previously-stitched map to a different image without any error.
    """
    return Path(embedding_path).stem + "_microsam_features.npz"


# Same store -> same .npz, so a name collision would mean two samples silently
# sharing one feature map. Fail loudly instead.
_names = [_npz_name(p) for p in (_embedding_path(s) for s in SAMPLES) if p]
assert len(_names) == len(set(_names)), (
    f"Embedding stores map to duplicate cache names: "
    f"{[n for n in _names if _names.count(n) > 1]}"
)

data_list = []
for i, s in enumerate(SAMPLES):
    npz_path = None
    emb = _embedding_path(s)
    if emb is not None:
        if not emb.exists():
            raise FileNotFoundError(f"sample {i}: embedding store not found: {emb}")
        # Stitches the tiled .zarr in-process, or falls back to
        # `conda run -n MICROSAM_ENV` when this env's zarr cannot read the store
        # (e.g. a zarr v3 store). Cached: stitches at most once per store.
        npz_path = load_or_stitch_embeddings(
            str(emb), save_path=str(emb_dir),
            npz_name=_npz_name(emb), env=MICROSAM_ENV,
        )
    data = build_cell_graph_data(
        ais_labels[i], intensity_images[i], gt_labels=gt_labels[i],
        microsam_npz_path=npz_path, display_image=display_images[i], k=K,
        # 0.1, not the 0.5 default: the overlap distribution is bimodal and 0.5 cuts through
        # the populated real-cell mode, calling 27% of fragments background and deleting the
        # true edges they had to their cell-mates. Matches 12_Node Type GNN.ipynb.
        min_overlap_frac=0.1,
    )
    data_list.append(data)
    n_pos = int(data.edge_label.sum().item())
    print(f"graph {i}: {data.num_nodes} nodes, {data.edge_index.shape[1]} directed edges, "
          f"{n_pos} positive-edge entries, "
          f"visual={'yes' if npz_path else 'no'}")


## 3. Display original image, prediction, GT, and graph

Per image: the raw image, the AIS prediction, the GT masks, and the candidate graph with
positive (same-cell / merge) edges in green and negative candidates as faint dotted lines.

In [ ]:
import matplotlib.pyplot as plt
from skimage.color import label2rgb

# Same renderer the logged TensorBoard figures use, so this preview matches them.
# It dispatches on channel count and percentile-stretches each channel: a
# multi-channel display image is uint16, and imshow clips RGB input at 255, so
# passing it straight to imshow renders it almost entirely white.
from image_processing_tools.scene_graph_network.gnn_train import _imshow_microscopy


def plot_cell_graph(ax, display_img, data, draw_candidates=True):
    _imshow_microscopy(ax, display_img)
    cen = data.centroids.cpu().numpy()          # (N, 2) in (y, x)
    ei = data.edge_index.cpu().numpy()
    el = data.edge_label.cpu().numpy()
    for k in range(ei.shape[1]):
        u, v = ei[0, k], ei[1, k]
        y1, x1 = cen[u]
        y2, x2 = cen[v]
        if el[k] >= 0.5:
            ax.plot([x1, x2], [y1, y2], "-", color="lime", lw=2, zorder=2)
        elif draw_candidates:
            ax.plot([x1, x2], [y1, y2], ":", color="cyan", lw=0.7, alpha=0.5, zorder=1)
    ax.scatter(cen[:, 1], cen[:, 0], s=18, c="yellow", edgecolors="black", lw=0.4, zorder=3)
    ax.set_title("Graph (green = merge)")
    ax.axis("off")


for i in range(len(data_list)):
    fig, axs = plt.subplots(1, 4, figsize=(22, 6))
    disp = display_images[i]
    _imshow_microscopy(axs[0], disp); axs[0].set_title("Original"); axs[0].axis("off")
    axs[1].imshow(label2rgb(ais_labels[i], bg_label=0))
    axs[1].set_title(f"AIS prediction ({int(ais_labels[i].max())} frags)"); axs[1].axis("off")
    axs[2].imshow(label2rgb(gt_labels[i], bg_label=0))
    axs[2].set_title(f"GT ({int(gt_labels[i].max())} cells)"); axs[2].axis("off")
    plot_cell_graph(axs[3], disp, data_list[i])
    fig.suptitle(f"Image {i}")
    plt.tight_layout()
    plt.show()

node classification - false predictions, oversegmented masks, undersegmented masks, epithelial vs. hyphal cells (from area and aspect ratio) - all can be auto-generated from gt mask

## 4. Save the graph dataset as an on-disk object

Persist the list of PyG graphs to `DATASET_ROOT/processed/data.pt` via `save_pyg_dataset`,
then reload to confirm.

In [ ]:
from image_processing_tools.scene_graph_network.gnn_data import save_pyg_dataset, load_pyg_dataset

save_pyg_dataset(data_list, DATASET_ROOT)
print(f"Saved {len(data_list)} graphs to {DATASET_ROOT}/processed/data.pt")

reloaded = load_pyg_dataset(DATASET_ROOT)
print(f"Reloaded {len(reloaded)} graphs.")

## 5. Overfit test on one graph

Sanity check that the model can fit the merge labels of a single graph end-to-end (dims
8 / 10). A near-perfect AUC here confirms the features + labels are learnable before
committing to full cross-validation.

In [ ]:
from image_processing_tools.scene_graph_network.gnn_train import train_overfit_test

# Truthy, not `is not None`: an unset embedding is "" in the JSON, and "" is not None.
USE_VISUAL = all(s.get("embedding") for s in SAMPLES)

one_graph = [data_list[0]]

model_params = {
    "hidden_channels": 64,
    "dropout_p": 0.0,
    "use_visual_features": USE_VISUAL,
    "node_feature_dim": 8,
    "edge_feature_dim": 10,
    # visual-branch params (used only when use_visual_features=True):
    "d_visual": 32,
    "node_bbox_pad_frac": 0.1,
    "edge_box_margin_frac": 0.15,
    "edge_box_margin_floor": 20,
    "roi_output_size": 7,
}

results = train_overfit_test(
    dataset=one_graph,
    max_epochs=1000,
    batch_size=1,
    learning_rate=0.01,
    model_params=model_params,
    patience=500,
    neg_sample_ratio=1.0,
    experiment="merge_overfit_one_graph_k10_minFrac0_1",
    interpret_column_order="grouped",   # attribution heatmap column order: "grouped" (default) or "clustered"
    heatmap_sample_size=15,             # edges per class in Interpretation/..._sampled
    heatmap_seed=0,                     # fixed so the sampled rows are reproducible
)
print(results)

## 6. Cross-validation

Leave-one-out over the graphs: each fold trains on every image but one and is
scored on the held-out image, so the reported AUC is generalization to an unseen
image rather than fit. Needs at least two graphs.

Per fold, `_log_figures` writes the prediction overlays, the attention parallel
coordinates, both interpretation heatmaps and the 2x2 merge figure under
`output/cv_experiment/<root>/<experiment>/fold_<k>/`, plus the attention CSV and
the prediction graph as sidecar files.


In [ ]:
from image_processing_tools.scene_graph_network.gnn_train import n_fold_validation

# Leave-one-out: as many folds as graphs. Lower it if the dataset grows.
NUM_FOLDS = len(data_list)
assert NUM_FOLDS >= 2, f"Cross-validation needs >= 2 graphs, dataset has {NUM_FOLDS}."

USE_VISUAL = all(s.get("embedding") for s in SAMPLES)

model_params = {
    "hidden_channels": 64,
    "dropout_p": 0.0,
    "use_visual_features": USE_VISUAL,
    "node_feature_dim": 8,
    "edge_feature_dim": 10,
    # visual-branch params (used only when use_visual_features=True):
    "d_visual": 32,
    "node_bbox_pad_frac": 0.1,
    "edge_box_margin_frac": 0.15,
    "edge_box_margin_floor": 20,
    "roi_output_size": 7,
}

cv_results = n_fold_validation(
    dataset=data_list,
    num_folds=NUM_FOLDS,
    max_epochs=1000,
    batch_size=1,
    learning_rate=0.01,
    model_params=model_params,        # same architecture as the overfit test above
    patience=200,
    min_epoch=50,                     # suppress early stopping until the model is trained
                                      # enough that a high AUC is not numerical luck
    neg_sample_ratio=1.0,             # fixed positive:negative ratio, independent of graph size
    label_smoothing=0.1,
    experiment="merge_cv_k10_minFrac0_1",   # new subfolder; the old
                                            # merge_cv run used k=7 / 0.5
    interpret_column_order="grouped",
    heatmap_sample_size=15,
    heatmap_seed=0,
)

for i, r in enumerate(cv_results, start=1):
    print(f"fold {i}: AUC={r['test_auc']:.4f} acc={r['test_acc']:.4f} "
          f"loss={r['test_loss']:.4f} thresh={r['best_threshold']:.4f}")
